# LR Sweep Comparison — MobileNetV3-Small

Aggregates `metrics.json` from every completed run in the learning-rate sweep and generates side-by-side comparison charts.

Run automatically by `scripts/run_lr_sweep.py`, or manually:
```bash
papermill notebooks/lr_sweep_comparison.ipynb outputs/executed/lr_sweep_comparison.ipynb
```

In [ ]:
# Root directory to scan for metrics.json files.
# All sub-directories matching logs_glob_pattern are included.
logs_root        = "logs"
logs_glob_pattern = "mobilenet_v3_small-*/metrics.json"
output_dir        = "outputs/comparison"

In [ ]:
import json
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Ensure cwd is the repo root (same logic as master notebook)
if not Path("main.py").exists() and Path("../main.py").exists():
    os.chdir("..")

OUT = Path(output_dir)
OUT.mkdir(parents=True, exist_ok=True)

# ── Discover and load all metrics.json files ─────────────────────────────────
metrics_files = sorted(Path(logs_root).glob(logs_glob_pattern))
print(f"Found {len(metrics_files)} metrics files:")
for f in metrics_files:
    print(f"  {f}")

if not metrics_files:
    raise FileNotFoundError(
        f"No metrics.json found under '{logs_root}/{logs_glob_pattern}'. "
        "Run the LR sweep first: python scripts/run_lr_sweep.py"
    )

runs = []
for f in metrics_files:
    data = json.loads(f.read_text())
    runs.append(data)

# Sort by learning rate ascending
runs.sort(key=lambda r: r["config"]["learning_rate"])

print(f"\n{'LR':<12} {'Epochs':>6}  {'mAP':>7}  {'Rank-1':>7}  {'Rank-5':>7}  {'Time(s)':>9}")
print("-" * 58)
for r in runs:
    cfg  = r["config"]
    ev   = r["evaluation"]
    trn  = r["training"]
    cmc  = ev.get("cmc", {})
    mapp = ev.get("map")
    r1   = cmc.get("1")
    r5   = cmc.get("5")
    def fmt(v): return f"{v:.1f}%" if v is not None else "  n/a"
    print(f"{cfg['learning_rate']:<12}  {cfg['max_epochs']:>6}  {fmt(mapp):>7}  {fmt(r1):>7}  {fmt(r5):>7}  {trn['elapsed_seconds']:>9.0f}")

In [ ]:
# ── Chart 1: Rank-1 and mAP vs learning rate ─────────────────────────────────
lrs    = [r["config"]["learning_rate"] for r in runs]
rank1s = [r["evaluation"]["cmc"].get("1")  for r in runs]
rank5s = [r["evaluation"]["cmc"].get("5")  for r in runs]
maps   = [r["evaluation"].get("map")        for r in runs]
lr_labels = [str(lr) for lr in lrs]

has_eval = any(v is not None for v in rank1s)

if has_eval:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("MobileNetV3-Small — LR Sweep Evaluation Comparison", fontsize=13)

    # Line chart: metric vs LR (log scale x)
    ax = axes[0]
    if any(v is not None for v in rank1s):
        vals = [v if v is not None else float("nan") for v in rank1s]
        ax.plot(lr_labels, vals, "o-", label="Rank-1",  color="#2196F3", linewidth=2, markersize=8)
    if any(v is not None for v in rank5s):
        vals = [v if v is not None else float("nan") for v in rank5s]
        ax.plot(lr_labels, vals, "s-", label="Rank-5",  color="#4CAF50", linewidth=2, markersize=8)
    if any(v is not None for v in maps):
        vals = [v if v is not None else float("nan") for v in maps]
        ax.plot(lr_labels, vals, "^--", label="mAP",   color="#FF5722", linewidth=2, markersize=8)
    ax.set_xlabel("Learning Rate"); ax.set_ylabel("Score (%)")
    ax.set_title("Metric vs Learning Rate"); ax.legend(); ax.grid(True, alpha=0.3)
    ax.tick_params(axis="x", rotation=30)

    # Grouped bar chart: Rank-1 / mAP per LR
    ax = axes[1]
    x = range(len(lrs))
    w = 0.35
    r1_vals  = [v if v is not None else 0 for v in rank1s]
    map_vals = [v if v is not None else 0 for v in maps]

    bars1 = ax.bar([i - w/2 for i in x], r1_vals,  w, label="Rank-1", color="#2196F3", edgecolor="white")
    bars2 = ax.bar([i + w/2 for i in x], map_vals, w, label="mAP",    color="#FF5722", edgecolor="white")
    ax.set_xticks(list(x)); ax.set_xticklabels(lr_labels, rotation=30)
    ax.set_xlabel("Learning Rate"); ax.set_ylabel("Score (%)")
    ax.set_title("Rank-1 vs mAP per LR")
    ax.legend(); ax.grid(True, axis="y", alpha=0.3)

    for bar in bars1 + bars2:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.3,
                    f"{h:.1f}", ha="center", fontsize=7)

    plt.tight_layout()
    p = OUT / "lr_sweep_eval_comparison.png"
    plt.savefig(p, dpi=130, bbox_inches="tight")
    plt.show(); plt.close()
    print(f"Saved: {p}")
else:
    print("No evaluation metrics available yet — run experiments first.")

In [ ]:
# ── Chart 2: Training loss comparison across LR values ───────────────────────
# One line per LR run, x-axis = epoch, y-axis = total loss

has_training = any(len(r["training"]["epochs"]) > 0 for r in runs)

if has_training:
    colours = ["#E91E63", "#FF5722", "#4CAF50", "#2196F3", "#9C27B0"]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("MobileNetV3-Small — LR Sweep Training Comparison", fontsize=13)

    for ax_idx, (metric_key, ylabel, title) in enumerate([
        ("loss", "Total Loss",     "Training Loss (XEnt + Triplet)"),
        ("xent", "XEnt Loss",      "Cross-Entropy Loss"),
        ("acc",  "Top-1 Acc (%)",  "Training Accuracy"),
    ]):
        ax = axes[ax_idx]
        for i, r in enumerate(runs):
            epochs_data = r["training"]["epochs"]
            if not epochs_data:
                continue
            xs = [e["epoch"] for e in epochs_data]
            ys = [e[metric_key] for e in epochs_data]
            lr = r["config"]["learning_rate"]
            ax.plot(xs, ys, "o-", label=f"lr={lr}",
                    color=colours[i % len(colours)], linewidth=2, markersize=5)
        ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
        ax.set_title(title); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    p = OUT / "lr_sweep_training_comparison.png"
    plt.savefig(p, dpi=130, bbox_inches="tight")
    plt.show(); plt.close()
    print(f"Saved: {p}")
else:
    print("No per-epoch training data found.")

In [ ]:
# ── Chart 3: CMC curves overlaid ─────────────────────────────────────────────
any_cmc = any(r["evaluation"].get("cmc") for r in runs)

if any_cmc:
    colours = ["#E91E63", "#FF5722", "#4CAF50", "#2196F3", "#9C27B0"]

    fig, ax = plt.subplots(figsize=(8, 5))
    fig.suptitle("CMC Curves — LR Sweep", fontsize=13)

    for i, r in enumerate(runs):
        cmc = r["evaluation"].get("cmc", {})
        if not cmc:
            continue
        ranks  = sorted(int(k) for k in cmc)
        values = [cmc[str(k)] for k in ranks]
        lr     = r["config"]["learning_rate"]
        ax.plot(ranks, values, "o-", label=f"lr={lr}",
                color=colours[i % len(colours)], linewidth=2, markersize=6)

    ax.set_xlabel("Rank"); ax.set_ylabel("Matching Rate (%)")
    ax.legend(); ax.grid(True, alpha=0.3)
    ax.set_xticks(sorted({int(k) for r in runs for k in r["evaluation"].get("cmc", {})}))

    plt.tight_layout()
    p = OUT / "lr_sweep_cmc_comparison.png"
    plt.savefig(p, dpi=130, bbox_inches="tight")
    plt.show(); plt.close()
    print(f"Saved: {p}")
else:
    print("No CMC data found.")

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print("\nFull results table")
print("=" * 75)
header = f"{'LR':<12} {'mAP':>7}  {'R-1':>6}  {'R-5':>6}  {'R-10':>6}  {'R-20':>6}  {'Time(s)':>8}"
print(header)
print("-" * 75)

best_r1  = max((r["evaluation"]["cmc"].get("1",  0) or 0) for r in runs)
best_map = max((r["evaluation"].get("map",       0) or 0) for r in runs)

for r in runs:
    cfg = r["config"]
    ev  = r["evaluation"]
    cmc = ev.get("cmc", {})
    mapp = ev.get("map")
    def g(k): return cmc.get(str(k))
    def f(v): return f"{v:.1f}%" if v is not None else "  n/a"
    marker = " <-- best" if (g(1) or 0) >= best_r1 and best_r1 > 0 else ""
    print(
        f"{cfg['learning_rate']:<12}  {f(mapp):>7}  {f(g(1)):>6}  {f(g(5)):>6}  "
        f"{f(g(10)):>6}  {f(g(20)):>6}  {r['training']['elapsed_seconds']:>8.0f}"
        + marker
    )

print("-" * 75)
print(f"\nOutput charts in: {OUT.resolve()}")